In [1]:
import os, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- display settings ----
pd.set_option("display.max_columns", None)     # show all columns
pd.set_option("display.max_rows", 100)         # rows to show
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)
# pd.reset_option("all")                       # reset everything

print(pd.__version__)

2.3.3


In [2]:
# pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
X = online_retail.data.features 
y = online_retail.data.targets 
  
# metadata 
# print(online_retail.metadata) 
  
# variable information 
# print(online_retail.variables) 


In [4]:
X.head(2)

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,"17,850.00",United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom


In [5]:
df = online_retail.data.original
df.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,"17,850.00",United Kingdom


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## Handling Duplicates

In [7]:
df.duplicated().sum()
# df.duplicated(subset=["Description","Quantity","InvoiceDate"]).sum()

np.int64(5268)

In [8]:
# df[df.duplicated(subset=['InvoiceNo','StockCode'])]

In [9]:
# df[df.duplicated(subset=['InvoiceNo','StockCode'],keep='last')]

In [10]:
df_d = df.drop_duplicates(keep="last")

In [11]:
df_d.shape

(536641, 8)

In [12]:
df_null=df_d.isna().sum().reset_index()

In [13]:
df_null.columns=['Remarks','Count']
df_null[df_null['Count']>0]

,Remarks,Count
2,Description,1454
6,CustomerID,135037


In [14]:
#droping rows where unit price is 0 as it have no impact
df1=df[df['UnitPrice']!=0]
print(f"count of rows having dropping unit price is 0 {df[df['UnitPrice']==0].shape[0]}")

count of rows having dropping unit price is 0 2515


In [15]:
df1.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country'], dtype='object')

In [16]:
df_up=df[df['StockCode'].isin(list(set(df[df['UnitPrice']==0]['StockCode'])))].groupby(['StockCode','Description'])['UnitPrice'].agg('count').reset_index()

In [17]:
df_up.columns=['StockCode','Description','count']

In [18]:
df_up.head()

,StockCode,Description,count
0,10002,INFLATABLE POLITICAL GLOBE,71
1,10080,GROOVY CACTUS INFLATABLE,22
2,10080,check,1
3,10123C,HEARTS WRAPPING TAPE,3
4,10133,COLOURING PENCILS BROWN TUBE,199


In [19]:
# ----- detect -----
df.isnull().sum()                                     # count per column
(df.isnull().mean() * 100).round(2)                   # % missing
df.isnull().sum().sort_values(ascending=False)
df.isnull().any(axis=1).sum()                         # rows with at least one null
df.isnull().sum(axis=1).value_counts()                # how many nulls per row
df.notnull().all()                                    # columns with NO nulls

# summary table of missing data
missing = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct":   (df.isnull().mean() * 100).round(2),
    "dtype":      df.dtypes,
}).query("null_count > 0").sort_values("null_pct", ascending=False)
missing

,null_count,null_pct,dtype
CustomerID,135080,24.93,float64
Description,1454,0.27,object
